# 03 – Modeling
Goal: split the data, build a preprocessing pipeline, and train baseline models.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

from src.data import load_data
from src.features import NUMERIC_FEATURES, CATEGORICAL_FEATURES, TARGET

df = load_data("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")

X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = df[TARGET]
print(X.shape, y.shape)

(7043, 17) (7043,)


In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train:", X_train.shape, " Test:", X_test.shape)
print(f"Churn rate — train: {y_train.mean():.3f}, test: {y_test.mean():.3f}")

Train: (5634, 17)  Test: (1409, 17)
Churn rate — train: 0.265, test: 0.265


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), NUMERIC_FEATURES),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
])

In [4]:
X_train_transformed = preprocessor.fit_transform(X_train)
print(X_train_transformed.shape)

preprocessor.get_feature_names_out()[:10]

(5634, 42)


array(['num__tenure', 'num__MonthlyCharges', 'num__SeniorCitizen',
       'cat__Partner_No', 'cat__Partner_Yes', 'cat__Dependents_No',
       'cat__Dependents_Yes', 'cat__PhoneService_No',
       'cat__PhoneService_Yes', 'cat__MultipleLines_No'], dtype=object)

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc", "average_precision"]


def evaluate(model, name):
    pipe = Pipeline([("prep", preprocessor), ("model", model)])
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring)
    return pd.Series({m: scores[f"test_{m}"].mean() for m in scoring}, name=name)

In [6]:
results = pd.DataFrame([
    evaluate(DummyClassifier(strategy="most_frequent"), "Dummy (always 'No')"),
    evaluate(LogisticRegression(max_iter=1000), "Logistic Regression"),
]).round(3)

results

/Users/glitter/Glitter Cooking/customer-churn-model/venv/lib/python3.15/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/glitter/Glitter Cooking/customer-churn-model/venv/lib/python3.15/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/glitter/Glitter Cooking/customer-churn-model/venv/lib/python3.15/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.


,accuracy,precision,recall,f1,roc_auc,average_precision
Dummy (always 'No'),0.735,0.000,0.000,0.000,0.500,0.265
Logistic Regression,0.801,0.653,0.538,0.589,0.845,0.660


### Baseline results (5-fold CV on training set)
- The Dummy model gets 73.5% accuracy while catching 0% of churners → accuracy is misleading with imbalanced data.
- Logistic Regression catches 53.8% of churners (recall) with 65.3% precision, and PR-AUC of 0.660 vs 0.265 for the baseline.
- At the default 0.5 threshold, it still misses about 46.2% of churners (1 − recall). Each one costs ~$300 → Steps 5–6 will try to improve this with imbalance handling and a cost-based threshold.

## Step 5 – Handling class imbalance

In [7]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier


def evaluate_pipeline(pipe, name):
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring)
    return pd.Series({m: scores[f"test_{m}"].mean() for m in scoring}, name=name)


neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
ratio = neg / pos
print(f"Non-churners: {neg}, churners: {pos}, ratio: {ratio:.2f}")

Non-churners: 4139, churners: 1495, ratio: 2.77


In [8]:
more_results = pd.DataFrame([
    evaluate(
        LogisticRegression(max_iter=1000, class_weight="balanced"),
        "LogReg (balanced)",
    ),
    evaluate_pipeline(
        ImbPipeline([
            ("prep", preprocessor),
            ("smote", SMOTE(random_state=42)),
            ("model", LogisticRegression(max_iter=1000)),
        ]),
        "LogReg + SMOTE",
    ),
    evaluate(
        XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
                      random_state=42, eval_metric="logloss"),
        "XGBoost",
    ),
    evaluate(
        XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
                      scale_pos_weight=ratio, random_state=42, eval_metric="logloss"),
        "XGBoost (weighted)",
    ),
]).round(3)

all_results = pd.concat([results, more_results])
all_results.sort_values("average_precision", ascending=False)

,accuracy,precision,recall,f1,roc_auc,average_precision
XGBoost,0.800,0.658,0.518,0.579,0.844,0.666
XGBoost (weighted),0.758,0.530,0.782,0.632,0.844,0.665
Logistic Regression,0.801,0.653,0.538,0.589,0.845,0.660
LogReg (balanced),0.750,0.519,0.793,0.628,0.845,0.658
LogReg + SMOTE,0.760,0.533,0.774,0.631,0.844,0.658
Dummy (always 'No'),0.735,0.000,0.000,0.000,0.500,0.265


In [9]:
for name, model in [
    ("Logistic Regression", LogisticRegression(max_iter=1000)),
    ("XGBoost", XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
                              random_state=42, eval_metric="logloss")),
]:
    pipe = Pipeline([("prep", preprocessor), ("model", model)])
    s = cross_validate(pipe, X_train, y_train, cv=cv, scoring="average_precision")["test_score"]
    print(f"{name}: PR-AUC = {s.mean():.3f} ± {s.std():.3f}   (5 folds: {s.round(3)})")

Logistic Regression: PR-AUC = 0.660 ± 0.018   (5 folds: [0.645 0.635 0.683 0.676 0.661])
XGBoost: PR-AUC = 0.666 ± 0.018   (5 folds: [0.651 0.639 0.686 0.68  0.674])


- XGBoost was slightly higher in all 5 folds (consistent but small edge of ~0.006), not enough to outweigh Logistic Regression's simplicity and interpretability.